<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/SegmentationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q kagglehub
import kagglehub, pathlib, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
print("downloaded to:", path)

Using Colab cache for faster access to the 'rescuenet' dataset.
downloaded to: /kaggle/input/rescuenet


In [15]:
root = pathlib.Path(path)
org_dir   = list(root.rglob("train-org-img"))[0]
label_dir = list(root.rglob("train-label-img"))[0]
print("images:", len(list(org_dir.glob("*.jpg"))), "| masks:", len(list(label_dir.glob("*.png"))))

images: 3595 | masks: 3595


In [16]:
#0=Unlabeled, 1= Water, 2 = Building w/o Damage
#3 Building with minor damage, 4 Building with Major damage
#5 Building completely destroyed, 6 Vechicle, 7 Clear Road
#8 Blocked Road, 9 Tree, 10 Pool

!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp
import torch, torch.nn as nn, torch.optim as optim

In [17]:
class_number = 11

model = smp.Unet(encoder_name="resnet34",
                 encoder_weights="imagenet",
                 in_channels=3,
                 classes=class_number,
                 )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [18]:

train_original   = list(root.rglob("train-org-img"))[0]
train_label = list(root.rglob("train-label-img"))[0]
val_original     = list(root.rglob("val-org-img"))[0]
val_label   = list(root.rglob("val-label-img"))[0]

print("train imgs/masks:", len(list(train_original.glob('*.jpg'))), len(list(train_label.glob('*.png'))))
print("val imgs/masks:  ", len(list(val_original.glob('*.jpg'))), len(list(val_label.glob('*.png'))))

train imgs/masks: 3595 3595
val imgs/masks:   449 449


In [19]:
loss = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass") # diceloss is used common in image segmentation by focusing on the intersection of predicted mask and the real mask

def criterion(logits, masks):
  return loss(logits, masks) + dice_loss(logits, masks)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [20]:
import pathlib, numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

In [21]:
#RescueNetSegmentedDataset class, Kento

In [22]:
#build the data sets from org_dir, label_dir, see ResNetModel for reference, Oluj
#equivalent to class LadiDataset(Dataset):
import os
from pathlib import Path
class RescueNetDataset(Dataset):
    def __init__(self, image_dir, mask_dir, root_dir, transform):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        #print(transform)
        self.images = os.listdir(image_dir)
        self.root_dir_img = root_dir

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img_id = img_path.split('.')[0]
        img_path = os.path.join(self.root_dir_img, img_path)
        mask_path = os.path.join(self.mask_dir, f"{img_id}_lab.png")

        image =  Image.open(img_path).convert("RGB") #converted to RGB channels because it's n image
        mask = Image.open(mask_path) #leaving as single channel image with integer ids.

        image = np.array(image)
        mask = np.array(mask.convert("L"), dtype=np.int64)
        mask[mask == 11] = 10

        image = tv_tensors.Image(torch.from_numpy(image).permute(2,0,1)) #reshape and moves channels
        mask = tv_tensors.Mask(torch.from_numpy(mask))

        if self.transform is not None:
          image, mask = self.transform(image, mask)
          #print('applied transform')

        return(image, mask)

        #----------------------------------------------------------
        # image = image.resize((self.size,self.size), Image.BILINEAR) #resizing to chosen size, used BILINEAR for interpolation
        # mask = mask.resize((self.size,self.size), Image.NEAREST) #masks needs no averaging, so everythings stays as an integer

        # if self.train and torch.rand(1).item() < 0.5:
        #     image = TF. hflip(image); mask = TF.hflip(mask) #keeps any transformation that's applied to the image applied to the mask

        # image = TF.to_tensor(image)
        # image = TF.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) #this line and the line before converts the image into the numeric form the pretrained model was trained on,
        # #they convert a bunch of stuff like from PIL image to a PyTorch tensor, rescales pixel values down to 0-1 floats, etc.
        # mask = torch.as_tensor(np.array(mask), dtype = torch.long) #similar thing converts the mask to a long tensor shape
        # return{"image": image, "mask": mask} #returns the image and mask as Pytorch tensors, image is a float32, while mask is a long(no color channel dimension)


In [23]:
#IoU or mIOU evaluation function, Aishani
def calc_IOU(pred_arr, label_arr):
  #multiclass - each class has separate IOU score?
  iou_scores = []
  #union pix
  for c_index in range(class_number):
    #get where they overlap and are correct
    class_gt = label_arr == c_index
    class_pred_area = pred_arr == c_index
    intersect_pix = np.sum((class_gt & class_pred_area))
    intersect_pix = float(intersect_pix)

    #get total area of masks merged

    total_region = np.sum((class_gt | class_pred_area))
    total_region = float(total_region)

    if(total_region == 0):
      continue

    iou_scores.append(intersect_pix/total_region)

  return iou_scores

def calc_mean_IOU(iou_scores): # can merge into another output above
  #iou_scores = [score for score in iou_scores if score != 0]
  return np.mean(iou_scores)


#testing
# import PIL
# import os
# from pathlib import Path
# import matplotlib.pyplot as plt
# c = 0
# p_img_masks = []
# for f in Path(train_label).iterdir():
#   if c<5:
#     img = PIL.Image.open(f)
#     img_arr = np.array(img)
#     p_img_masks.append(img_arr)
#   c+=1

# print(calc_IOU(p_img_masks[4],p_img_masks[1]))
# fix ,ax = plt.subplots(1,2,figsize=(6,6))

# ax[0].imshow(p_img_masks[4])
# ax[1].imshow(p_img_masks[1])

In [24]:
import torch
import torchvision
from torch.utils.data import DataLoader

def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])

def get_loaders(train_dir, train_mask_dir, val_dir, val_mask_dir, batch_size, train_transform, val_transform, num_workers=0, pin_memory=True):
    train_img_dir = os.path.join(path, "RescueNet", "train", "train-org-img")
    # train_mask_dir = os.path.join(path, "RescueNet", "train", "train-label-img")
    val_img_dir = os.path.join(path, "RescueNet", "val", "val-org-img")
    # val_mask_dir = os.path.join(path, "RescueNet", "val", "val-label-img")

    #print('TRAIN LOADER')
    train_dataset = RescueNetDataset(image_dir=train_dir, mask_dir = train_mask_dir, root_dir = train_img_dir, transform=train_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=True)

    #print('VAL LOADER')
    val_dataset = RescueNetDataset(image_dir = val_dir, mask_dir = val_mask_dir, root_dir = val_img_dir, transform=val_transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory)

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            #evaluation of model
            preds = torch.argmax(model(x), dim=1)
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)

    print(f"Got {num_correct}/{num_pixels} with accuracy {num_correct/num_pixels*100:.2f}")
    calc_iou_var = calc_IOU(preds, y)
    print(f"Calc IoU: {calc_iou_var}")
    print(f"Calc Mean IoU: {calc_mean_IOU(calc_iou_var)}")
    model.train()

In [ ]:
#training, saving checkpoints to drive, Brian
import torch
from torchvision.transforms import v2
from torchvision import tv_tensors
from tqdm import tqdm
import os
import torch.nn as nn
import torch.optim as optim

# path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")

#hyperparameters

alpha = 1e-4 #learning rate
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 4
num_epochs = 3
num_workers = 0
image_height =160
image_width = 240
pin_memory = False
load_model = True
# train_img_dir = os.path.join(path, "RescueNet", "train", "train-org-img")
# train_mask_dir = os.path.join(path, "RescueNet", "train", "train-label-img")
# val_img_dir = os.path.join(path, "RescueNet", "val", "val-org-img")
# val_mask_dir = os.path.join(path, "RescueNet", "val", "val-label-img")

def train(loader, model, optimizer, loss_func, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        # print("Batch index:", batch_idx)
        # print("Data shape:", data.shape)
        # print("Targets shape:", targets.shape)

        data = data.to(device=device)
        #print("data moved")
        targets = targets.to(device=device)
        #print("target moved")

        with torch.amp.autocast(device_type=device):
            predictions = model(data)
            # print(predictions.shape)
            # print(targets.shape)
            loss = loss_func(predictions, targets)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=loss.item())

        del data, targets, predictions, loss

def main():
    train_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.RandomRotation(degrees=35),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.1),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
    ])

    val_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
    ])

    loss_func = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=alpha)

    train_loader, val_loader = get_loaders(train_original, train_label,
                                           val_original, val_label,
                                           batch_size,
                                           train_transforms, val_transforms,
                                           num_workers, pin_memory)

    #CHECK LOADER
    data, targets = next(iter(train_loader))
    # print(targets.max())
    # print(targets.min())
    # print(torch.unique(targets))
    # data, targets = next(iter(train_loader))

    scaler = torch.amp.GradScaler()

    for epoch in range(num_epochs):
        train(train_loader, model, optimizer, loss_func, scaler)

        checkpoint = {"state_dict": model.state_dict(),
                      "optimizer": optimizer.state_dict()}
        save_checkpoint(checkpoint)

        check_accuracy(val_loader, model, device=device)
        torch.cuda.empty_cache()

if __name__ == "__main__":
  print("Training has begun!")
  main()

Training has begun!


 62%|██████▏   | 553/899 [14:16<08:50,  1.53s/it, loss=1.01]